In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS utilities_silver_location
URL 'abfss://silver@utilitiesmaintainence.dfs.core.windows.net'
WITH (STORAGE CREDENTIAL utilities_adls_credential);

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS motordatabricks.utilities_silver;

In [0]:
%sql
GRANT USE SCHEMA ON SCHEMA motordatabricks.utilities_silver TO `1833017mdcs@cit.edu.in`;

In [0]:
%sql
GRANT CREATE TABLE ON SCHEMA motordatabricks.utilities_silver TO `1833017mdcs@cit.edu.in`;

In [0]:
%sql
GRANT READ FILES, WRITE FILES, CREATE EXTERNAL TABLE ON EXTERNAL LOCATION utilities_silver_location
TO `1833017mdcs@cit.edu.in`;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS motordatabricks.utilities_silver.maintenance (
    work_order_id STRING,
    asset_id STRING,
    technician_id STRING,
    location_id STRING,

    maintenance_type STRING,
    maintenance_category STRING,
    planned_flag STRING,
    failure_type STRING,
    failure_severity STRING,
    priority STRING,
    work_order_status STRING,
    resolution_code STRING,

    created_date TIMESTAMP,
    scheduled_date TIMESTAMP,
    completed_date TIMESTAMP,

    downtime_hours DOUBLE,
    labor_hours DOUBLE,
    parts_used INT,
    parts_cost DOUBLE,
    labor_cost DOUBLE,
    total_cost DOUBLE,

    source_system STRING,
    updated_at TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@utilitiesmaintainence.dfs.core.windows.net/maintenance/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS motordatabricks.utilities_silver.assets (
    asset_id STRING,
    asset_type STRING,
    manufacturer STRING,
    model STRING,
    serial_number STRING,
    installation_date DATE,
    location_id STRING,
    city STRING,
    region STRING,
    criticality STRING,
    asset_status STRING
)
USING DELTA
LOCATION 'abfss://silver@utilitiesmaintainence.dfs.core.windows.net/assets/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS motordatabricks.utilities_silver.locations (
    location_id STRING,
    city STRING,
    region STRING,
    site_name STRING,
    site_type STRING
)
USING DELTA
LOCATION 'abfss://silver@utilitiesmaintainence.dfs.core.windows.net/locations/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS motordatabricks.utilities_silver.technicians (
    technician_id STRING,
    technician_name STRING,
    skill_level STRING,
    team STRING,
    experience_years INT
)
USING DELTA
LOCATION 'abfss://silver@utilitiesmaintainence.dfs.core.windows.net/technicians/';

In [0]:
%sql
SHOW TABLES IN motordatabricks.utilities_silver;

database,tableName,isTemporary
utilities_silver,assets,false
utilities_silver,locations,false
utilities_silver,maintenance,false
utilities_silver,technicians,false


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce

# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "utilitiesmaintainence"

bronze_base_path = (
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
)

maintenance_base_path = bronze_base_path + "monthly_files/"
reference_base_path = bronze_base_path + "reference/"


# ============================================================
# 2. EXPECTED SILVER MAINTENANCE SCHEMA
# ============================================================

expected_types = {
    "work_order_id": "string",
    "asset_id": "string",
    "technician_id": "string",
    "location_id": "string",

    "maintenance_type": "string",
    "maintenance_category": "string",
    "planned_flag": "string",
    "failure_type": "string",
    "failure_severity": "string",
    "priority": "string",
    "work_order_status": "string",
    "resolution_code": "string",

    "created_date": "timestamp",
    "scheduled_date": "timestamp",
    "completed_date": "timestamp",

    "downtime_hours": "double",
    "labor_hours": "double",
    "parts_used": "int",
    "parts_cost": "double",
    "labor_cost": "double",
    "total_cost": "double",

    "source_system": "string",
    "updated_at": "timestamp"
}

expected_columns = list(expected_types.keys())

# These columns are mandatory for a valid work order
mandatory_columns = [
    "work_order_id",
    "asset_id",
    "technician_id",
    "location_id"
]


# ============================================================
# 3. READ JANUARY, FEBRUARY, MARCH
# ============================================================

months = ["01", "02", "03"]

valid_monthly_dfs = []
schema_results = []

for month in months:

    file_name = f"maintenance_2026_{month}.csv"
    file_path = maintenance_base_path + file_name

    print(f"\n========== Processing {file_name} ==========")

    # Read as strings first.
    # This gives us better control over future schema/type changes.
    raw_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(file_path)
    )

    actual_columns = raw_df.columns

    # --------------------------------------------------------
    # Check required columns
    # --------------------------------------------------------

    missing_required = [
        c for c in mandatory_columns
        if c not in actual_columns
    ]

    extra_columns = [
        c for c in actual_columns
        if c not in expected_columns
    ]

    if missing_required:
        print(
            f"REJECTED: {file_name} "
            f"missing mandatory columns: {missing_required}"
        )

        schema_results.append(
            (
                file_name,
                "REJECTED",
                str(missing_required),
                str(extra_columns)
            )
        )

        # Do not allow a bad file into Silver
        continue

    # --------------------------------------------------------
    # Add missing optional Silver columns as NULL
    # --------------------------------------------------------

    df = raw_df

    for column_name, data_type in expected_types.items():

        if column_name not in df.columns:

            print(
                f"Optional column missing in {file_name}: "
                f"{column_name} -> adding NULL"
            )

            df = df.withColumn(
                column_name,
                F.lit(None).cast(data_type)
            )

    # --------------------------------------------------------
    # Keep expected Silver columns only
    # --------------------------------------------------------

    df = df.select(expected_columns)

    # --------------------------------------------------------
    # Convert expected data types
    # --------------------------------------------------------

    for column_name, data_type in expected_types.items():

        if data_type != "string":

            df = df.withColumn(
                column_name,
                F.col(column_name).cast(data_type)
            )

    # --------------------------------------------------------
    # Add source file for lineage/debugging
    # This is kept temporarily for DQ and debugging.
    # We will decide tomorrow whether to persist it in Silver.
    # --------------------------------------------------------

    df = df.withColumn(
        "_source_file",
        F.lit(file_name)
    )

    valid_monthly_dfs.append(df)

    schema_results.append(
        (
            file_name,
            "VALID",
            "[]",
            str(extra_columns)
        )
    )


# ============================================================
# 4. DISPLAY SCHEMA VALIDATION RESULTS
# ============================================================

schema_result_df = spark.createDataFrame(
    schema_results,
    [
        "file_name",
        "schema_status",
        "missing_mandatory_columns",
        "extra_columns"
    ]
)

display(schema_result_df)


# ============================================================
# 5. STOP IF NONE OF THE FILES ARE VALID
# ============================================================

if not valid_monthly_dfs:

    raise Exception(
        "No valid maintenance files available for Silver processing."
    )


# ============================================================
# 6. UNION ALL VALID MONTHLY FILES
# ============================================================

maintenance_all_df = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    valid_monthly_dfs
)

print(
    "Total records before cleaning:",
    maintenance_all_df.count()
)


# ============================================================
# 7. CLEAN STRING COLUMNS
# ============================================================

string_columns = [
    column_name
    for column_name, data_type in expected_types.items()
    if data_type == "string"
]

clean_df = maintenance_all_df

for column_name in string_columns:

    clean_df = clean_df.withColumn(
        column_name,
        F.when(
            F.trim(F.col(column_name)) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col(column_name))
        )
    )


# ============================================================
# 8. STANDARDIZE PLANNED FLAG
# ============================================================

clean_df = clean_df.withColumn(
    "planned_flag",
    F.when(
        F.upper(F.col("planned_flag")).isin("YES", "Y", "TRUE", "1"),
        F.lit("Y")
    )
    .when(
        F.upper(F.col("planned_flag")).isin("NO", "N", "FALSE", "0"),
        F.lit("N")
    )
    .otherwise(
        F.upper(F.col("planned_flag"))
    )
)


# ============================================================
# 9. STANDARDIZE CATEGORICAL VALUES
# ============================================================

categorical_columns = [
    "maintenance_type",
    "maintenance_category",
    "failure_type",
    "failure_severity",
    "priority",
    "work_order_status",
    "resolution_code",
    "source_system"
]

for column_name in categorical_columns:

    clean_df = clean_df.withColumn(
        column_name,
        F.when(
            F.col(column_name).isNull(),
            F.lit(None)
        ).otherwise(
            F.upper(F.col(column_name))
        )
    )


# ============================================================
# 10. NULL DATA QUALITY CHECKS
# ============================================================

dq_results = []

for column_name in mandatory_columns:

    null_count = (
        clean_df
        .filter(F.col(column_name).isNull())
        .count()
    )

    dq_results.append(
        (
            "NULL_CHECK",
            column_name,
            null_count,
            "PASS" if null_count == 0 else "FAIL"
        )
    )


# ============================================================
# 11. DUPLICATE CHECK BEFORE DEDUPLICATION
# ============================================================

duplicate_groups_df = (
    clean_df
    .groupBy("work_order_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_group_count = duplicate_groups_df.count()

dq_results.append(
    (
        "DUPLICATE_CHECK",
        "work_order_id",
        duplicate_group_count,
        "PASS"
    )
)

print(
    "Duplicate work_order_id groups before dedup:",
    duplicate_group_count
)


# ============================================================
# 12. DETERMINISTIC DEDUPLICATION
# ============================================================

# If the same work_order_id appears more than once,
# keep the latest version using updated_at.

window_spec = (
    Window
    .partitionBy("work_order_id")
    .orderBy(
        F.col("updated_at").desc_nulls_last()
    )
)

dedup_df = (
    clean_df
    .withColumn("_row_number", F.row_number().over(window_spec))
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

print(
    "Records after deduplication:",
    dedup_df.count()
)


# ============================================================
# 13. NUMERIC DATA QUALITY CHECKS
# ============================================================

numeric_non_negative_columns = [
    "downtime_hours",
    "labor_hours",
    "parts_used",
    "parts_cost",
    "labor_cost",
    "total_cost"
]

for column_name in numeric_non_negative_columns:

    invalid_count = (
        dedup_df
        .filter(
            F.col(column_name).isNotNull()
            & (F.col(column_name) < 0)
        )
        .count()
    )

    dq_results.append(
        (
            "NON_NEGATIVE_CHECK",
            column_name,
            invalid_count,
            "PASS" if invalid_count == 0 else "FAIL"
        )
    )


# ============================================================
# 14. DATE/TIMESTAMP DATA QUALITY CHECKS
# ============================================================

invalid_completed_date_count = (
    dedup_df
    .filter(
        F.col("completed_date").isNotNull()
        & F.col("created_date").isNotNull()
        & (
            F.col("completed_date")
            < F.col("created_date")
        )
    )
    .count()
)

dq_results.append(
    (
        "DATE_CHECK",
        "completed_date >= created_date",
        invalid_completed_date_count,
        "PASS"
        if invalid_completed_date_count == 0
        else "FAIL"
    )
)


# ============================================================
# 15. REFERENTIAL INTEGRITY
# ============================================================

# Assets
assets_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(reference_base_path + "assets.csv")
)

# Locations
locations_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(reference_base_path + "locations.csv")
)

# Technicians
technicians_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(reference_base_path + "technicians.csv")
)


# Clean reference keys
assets_keys = (
    assets_df
    .select(
        F.trim(F.col("asset_id")).alias("asset_id")
    )
    .dropDuplicates()
)

locations_keys = (
    locations_df
    .select(
        F.trim(F.col("location_id")).alias("location_id")
    )
    .dropDuplicates()
)

technicians_keys = (
    technicians_df
    .select(
        F.trim(F.col("technician_id")).alias("technician_id")
    )
    .dropDuplicates()
)


# ------------------------------------------------------------
# Asset referential integrity
# ------------------------------------------------------------

invalid_asset_count = (
    dedup_df
    .select("asset_id")
    .filter(F.col("asset_id").isNotNull())
    .join(
        assets_keys,
        "asset_id",
        "left_anti"
    )
    .count()
)

dq_results.append(
    (
        "REFERENTIAL_INTEGRITY",
        "asset_id",
        invalid_asset_count,
        "PASS" if invalid_asset_count == 0 else "FAIL"
    )
)


# ------------------------------------------------------------
# Location referential integrity
# ------------------------------------------------------------

invalid_location_count = (
    dedup_df
    .select("location_id")
    .filter(F.col("location_id").isNotNull())
    .join(
        locations_keys,
        "location_id",
        "left_anti"
    )
    .count()
)

dq_results.append(
    (
        "REFERENTIAL_INTEGRITY",
        "location_id",
        invalid_location_count,
        "PASS" if invalid_location_count == 0 else "FAIL"
    )
)


# ------------------------------------------------------------
# Technician referential integrity
# ------------------------------------------------------------

invalid_technician_count = (
    dedup_df
    .select("technician_id")
    .filter(F.col("technician_id").isNotNull())
    .join(
        technicians_keys,
        "technician_id",
        "left_anti"
    )
    .count()
)

dq_results.append(
    (
        "REFERENTIAL_INTEGRITY",
        "technician_id",
        invalid_technician_count,
        "PASS" if invalid_technician_count == 0 else "FAIL"
    )
)


# ============================================================
# 16. BUSINESS DATA QUALITY CHECKS
# ============================================================

# In Progress work orders should not have completed_date
invalid_in_progress_count = (
    dedup_df
    .filter(
        (F.upper(F.col("work_order_status")) == "IN PROGRESS")
        & F.col("completed_date").isNotNull()
    )
    .count()
)

dq_results.append(
    (
        "BUSINESS_RULE",
        "IN PROGRESS should not have completed_date",
        invalid_in_progress_count,
        "PASS"
        if invalid_in_progress_count == 0
        else "FAIL"
    )
)


# ============================================================
# 17. CREATE DQ RESULT DATAFRAME
# ============================================================

dq_df = spark.createDataFrame(
    dq_results,
    [
        "check_type",
        "check_name",
        "issue_count",
        "status"
    ]
)

print("\n========== DATA QUALITY RESULTS ==========")

display(dq_df)


# ============================================================
# 18. FINAL DQ SUMMARY
# ============================================================

failed_checks = (
    dq_df
    .filter(F.col("status") == "FAIL")
    .count()
)

print("Failed DQ checks:", failed_checks)


# ============================================================
# 19. FINAL SILVER DATAFRAME
# ============================================================

silver_maintenance_df = (
    dedup_df
    .drop("_source_file")
)

print(
    "Final Silver maintenance records:",
    silver_maintenance_df.count()
)

print(
    "Final Silver columns:",
    silver_maintenance_df.columns
)

display(silver_maintenance_df)


========== Processing maintenance_2026_01.csv ==========

========== Processing maintenance_2026_02.csv ==========

========== Processing maintenance_2026_03.csv ==========


file_name,schema_status,missing_mandatory_columns,extra_columns
maintenance_2026_01.csv,VALID,[],"['asset_type', 'manufacturer', 'asset_model', 'asset_criticality', 'city', 'region', 'technician_name', 'technician_skill', 'maintenance_team']"
maintenance_2026_02.csv,VALID,[],"['asset_type', 'manufacturer', 'asset_model', 'asset_criticality', 'city', 'region', 'technician_name', 'technician_skill', 'maintenance_team']"
maintenance_2026_03.csv,VALID,[],"['asset_type', 'manufacturer', 'asset_model', 'asset_criticality', 'city', 'region', 'technician_name', 'technician_skill', 'maintenance_team']"


Total records before cleaning: 1800
Duplicate work_order_id groups before dedup: 0
Records after deduplication: 1800

========== DATA QUALITY RESULTS ==========


check_type,check_name,issue_count,status
NULL_CHECK,work_order_id,0,PASS
NULL_CHECK,asset_id,0,PASS
NULL_CHECK,technician_id,0,PASS
NULL_CHECK,location_id,0,PASS
DUPLICATE_CHECK,work_order_id,0,PASS
NON_NEGATIVE_CHECK,downtime_hours,0,PASS
NON_NEGATIVE_CHECK,labor_hours,0,PASS
NON_NEGATIVE_CHECK,parts_used,0,PASS
NON_NEGATIVE_CHECK,parts_cost,0,PASS
NON_NEGATIVE_CHECK,labor_cost,0,PASS


Failed DQ checks: 0
Final Silver maintenance records: 1800
Final Silver columns: ['work_order_id', 'asset_id', 'technician_id', 'location_id', 'maintenance_type', 'maintenance_category', 'planned_flag', 'failure_type', 'failure_severity', 'priority', 'work_order_status', 'resolution_code', 'created_date', 'scheduled_date', 'completed_date', 'downtime_hours', 'labor_hours', 'parts_used', 'parts_cost', 'labor_cost', 'total_cost', 'source_system', 'updated_at']


work_order_id,asset_id,technician_id,location_id,maintenance_type,maintenance_category,planned_flag,failure_type,failure_severity,priority,work_order_status,resolution_code,created_date,scheduled_date,completed_date,downtime_hours,labor_hours,parts_used,parts_cost,labor_cost,total_cost,source_system,updated_at
WO-202601-00001,AST-00138,TECH-0017,LOC-0011,PREVENTIVE,ELECTRICAL,Y,NO FAILURE,NONE,LOW,COMPLETED,ADJUSTED,2026-01-23T14:46:00.000Z,2026-01-26T14:46:00.000Z,2026-01-26T20:16:21.000Z,6.43,8.85,2,1091.55,778.51,1870.06,EAM,2026-01-26T20:16:21.000Z
WO-202601-00002,AST-00137,TECH-0100,LOC-0012,PREVENTIVE,MECHANICAL,Y,NO FAILURE,NONE,MEDIUM,COMPLETED,INSPECTED - NO ISSUE,2026-01-13T02:34:30.000Z,2026-01-19T02:34:30.000Z,2026-01-19T05:55:28.000Z,3.75,2.28,0,1206.57,224.53,1431.1,EAM,2026-01-19T05:55:28.000Z
WO-202601-00003,AST-00236,TECH-0095,LOC-0069,INSPECTION,INSPECTION,Y,NO FAILURE,NONE,LOW,COMPLETED,REPAIRED,2026-01-25T03:33:42.000Z,2026-01-26T03:33:42.000Z,2026-01-26T06:03:02.000Z,2.89,9.11,7,156.05,708.3,864.35,FIELD_SERVICE,2026-01-26T06:03:02.000Z
WO-202601-00004,AST-00153,TECH-0026,LOC-0065,PREVENTIVE,CONTROL SYSTEM,Y,NO FAILURE,NONE,LOW,COMPLETED,REPAIRED,2026-01-12T01:55:42.000Z,2026-01-13T01:55:42.000Z,2026-01-13T05:22:37.000Z,7.96,5.68,7,1041.09,321.29,1362.38,FIELD_SERVICE,2026-01-13T05:22:37.000Z
WO-202601-00005,AST-00306,TECH-0020,LOC-0071,CORRECTIVE,INSPECTION,N,INSULATION DAMAGE,CRITICAL,HIGH,COMPLETED,PREVENTIVE MAINTENANCE COMPLETED,2026-01-12T02:15:10.000Z,2026-01-14T02:15:10.000Z,2026-01-14T11:13:15.000Z,24.0,20.83,7,1067.86,1822.37,2890.23,FIELD_SERVICE,2026-01-14T11:13:15.000Z
WO-202601-00006,AST-00099,TECH-0003,LOC-0002,PREVENTIVE,STRUCTURAL,Y,NO FAILURE,NONE,MEDIUM,COMPLETED,ADJUSTED,2026-01-23T12:22:46.000Z,2026-01-25T12:22:46.000Z,2026-01-25T16:44:19.000Z,6.34,8.48,4,1368.75,693.85,2062.6,FIELD_SERVICE,2026-01-25T16:44:19.000Z
WO-202601-00007,AST-00469,TECH-0071,LOC-0022,PREVENTIVE,CONTROL SYSTEM,Y,NO FAILURE,NONE,MEDIUM,COMPLETED,ADJUSTED,2026-01-06T02:33:09.000Z,2026-01-12T02:33:09.000Z,2026-01-12T04:38:43.000Z,6.96,19.07,3,1416.83,1341.72,2758.55,CMMS,2026-01-12T04:38:43.000Z
WO-202601-00008,AST-00370,TECH-0009,LOC-0056,PREVENTIVE,ELECTRICAL,Y,NO FAILURE,NONE,MEDIUM,COMPLETED,REPAIRED,2026-01-22T17:52:24.000Z,2026-01-25T17:52:24.000Z,2026-01-25T23:45:19.000Z,9.42,8.23,2,546.98,531.56,1078.54,EAM,2026-01-25T23:45:19.000Z
WO-202601-00009,AST-00448,TECH-0005,LOC-0028,PREVENTIVE,ELECTRICAL,Y,NO FAILURE,NONE,MEDIUM,COMPLETED,REPAIRED,2026-01-07T03:31:12.000Z,2026-01-10T03:31:12.000Z,2026-01-10T07:19:47.000Z,10.2,13.57,3,2007.97,1113.81,3121.78,FIELD_SERVICE,2026-01-10T07:19:47.000Z
WO-202601-00010,AST-00252,TECH-0011,LOC-0012,CORRECTIVE,ELECTRICAL,N,CORROSION,HIGH,CRITICAL,IN PROGRESS,null,2026-01-15T20:03:23.000Z,2026-01-16T20:03:23.000Z,null,5.87,21.62,1,6360.87,1947.26,8308.13,FIELD_SERVICE,2026-01-16T23:55:06.000Z


In [0]:
# ============================================================
# STEP 1 — PREPARE SILVER REFERENCE DATAFRAMES
# ============================================================

# -------------------------
# Assets
# -------------------------

silver_assets_df = (
    assets_df
    .select(
        "asset_id",
        "asset_type",
        "manufacturer",
        "model",
        "serial_number",
        "installation_date",
        "location_id",
        "city",
        "region",
        "criticality",
        "asset_status"
    )
    .dropDuplicates(["asset_id"])
)


# -------------------------
# Locations
# -------------------------

silver_locations_df = (
    locations_df
    .select(
        "location_id",
        "city",
        "region",
        "site_name",
        "site_type"
    )
    .dropDuplicates(["location_id"])
)


# -------------------------
# Technicians
# -------------------------

silver_technicians_df = (
    technicians_df
    .select(
        "technician_id",
        "technician_name",
        "skill_level",
        "team",
        "experience_years"
    )
    .dropDuplicates(["technician_id"])
)


# ============================================================
# VERIFY RECORD COUNTS
# ============================================================

print("Maintenance:", silver_maintenance_df.count())
print("Assets:", silver_assets_df.count())
print("Locations:", silver_locations_df.count())
print("Technicians:", silver_technicians_df.count())

Maintenance: 1800
Assets: 500
Locations: 72
Technicians: 100


In [0]:
from delta.tables import DeltaTable

silver_assets_table = DeltaTable.forName(
    spark,
    "motordatabricks.utilities_silver.assets"
)

(
    silver_assets_table.alias("target")
    .merge(
        silver_assets_df.alias("source"),
        "target.asset_id = source.asset_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Assets MERGE completed successfully.")

Assets MERGE completed successfully.


In [0]:
from delta.tables import DeltaTable

# ============================================================
# STEP 2 — MERGE LOCATIONS, TECHNICIANS, MAINTENANCE
# ============================================================

# ------------------------------------------------------------
# 1. LOCATIONS MERGE
# ------------------------------------------------------------

silver_locations_table = DeltaTable.forName(
    spark,
    "motordatabricks.utilities_silver.locations"
)

(
    silver_locations_table.alias("target")
    .merge(
        silver_locations_df.alias("source"),
        "target.location_id = source.location_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Locations MERGE completed.")


# ------------------------------------------------------------
# 2. TECHNICIANS MERGE
# ------------------------------------------------------------

silver_technicians_table = DeltaTable.forName(
    spark,
    "motordatabricks.utilities_silver.technicians"
)

(
    silver_technicians_table.alias("target")
    .merge(
        silver_technicians_df.alias("source"),
        "target.technician_id = source.technician_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Technicians MERGE completed.")


# ------------------------------------------------------------
# 3. MAINTENANCE MERGE
# ------------------------------------------------------------

silver_maintenance_table = DeltaTable.forName(
    spark,
    "motordatabricks.utilities_silver.maintenance"
)

(
    silver_maintenance_table.alias("target")
    .merge(
        silver_maintenance_df.alias("source"),
        "target.work_order_id = source.work_order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Maintenance MERGE completed.")


print("========================================")
print("ALL 3 SILVER MERGES COMPLETED")
print("========================================")

Locations MERGE completed.
Technicians MERGE completed.
Maintenance MERGE completed.
ALL 3 SILVER MERGES COMPLETED
